
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# Shuffle

Shuffle is a Spark mechanism that redistributes data so that it's grouped differently across partitions. This typically involves copying data across executors and machines and, while it's sometimes necessary, it can be a complex and costly operation.

In this demo we will see shuffle in action. Run the next cell to set up the lesson.

In [0]:
%run ./Includes/Classroom-Setup-04.3

Run the following cell, which will set a Spark configuration variable that disables caching. Turning caching off makes the effect of the optimizations more apparent.

In [0]:
spark.conf.set('spark.databricks.io.cache.enabled', False)


### Data Creation

Let's generate the data we will use in this demo. First we'll synthesize data representing a set of sales transactions.

In [0]:
from pyspark.sql.functions import *

transactions_df = (spark
                        .range(0, 150000000, 1, 32)
                        .select(
                            'id',
                            round(rand() * 10000, 2).alias('amount'),
                            (col('id') % 10).alias('country_id'),
                            (col('id') % 100).alias('store_id')
                        )
                    )

transactions_df.display()

Now we'll write the data to a table.

In [0]:
transactions_df.write.mode('overwrite').saveAsTable('transactions')


Now let's synthesize data and write it to a table describing points of sale.

In [0]:
stores_df = (spark
                .range(0, 99)
                .select(
                    'id',
                    round(rand() * 100, 0).alias('employees'),
                    (col('id') % 10).alias('country_id'),
                    expr('uuid()').alias('name')
                )
            )

stores_df.display()

In [0]:
stores_df.write.saveAsTable('stores')

Now let's create a lookup table that maps `country_id` from the data tables to an actual country name.

In [0]:
countries = [(0, "Italy"),
             (1, "Canada"),
             (2, "Mexico"),
             (3, "China"),
             (4, "Germany"),
             (5, "UK"),
             (6, "Japan"),
             (7, "Korea"),
             (8, "Australia"),
             (9, "France"),
             (10, "Spain"),
             (11, "USA")
            ]

columns = ["id", "name"]
countries_df = spark.createDataFrame(data = countries, schema = columns)

countries_df.display()

In [0]:
countries_df.write.saveAsTable("countries")

## Joins

Now we'll perform a query that induces shuffling by joining three tables together, writing the results to a separate table.

**PLEASE NOTE:** in this cell, we're explicitly turning off broadcast joins in order to demonstrate shuffle.

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.databricks.adaptive.autoBroadcastJoinThreshold", -1)

joined_df = spark.sql("""
    SELECT 
        transactions.id,
        amount,
        countries.name as country_name,
        employees,
        stores.name as store_name
    FROM
        transactions
    LEFT JOIN
        stores
        ON
            transactions.store_id = stores.id
    LEFT JOIN
        countries
        ON
            transactions.country_id = countries.id
""")

joined_df.write.mode('overwrite').saveAsTable('transact_countries')

Open the Spark UI to the Stages page and notice there were two 1.4GB shuffle reads: one for each join:
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://petecurriculumstorage.blob.core.windows.net/images/join_shuffle.png" alt="Shuffle Join Stages" width="1000">
</div>

## Broadcast Join

Broadcast join avoids the shuffle. In the above cells we explicitly turned off broadcast joins, but now we'll return the configuration to the default so that broadcast join is enabled.  It only works in this case because at least one of the tables in each join are relatively small (< 100MB).

In [0]:
# Use default config & let broadcast join happen
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
spark.conf.unset("spark.databricks.adaptive.autoBroadcastJoinThreshold")

joined_df = spark.sql("""
    SELECT 
        transactions.id,
        amount,
        countries.name as country_name,
        employees,
        stores.name as store_name
    FROM
        transactions
    LEFT JOIN
        stores
        ON
            transactions.store_id = stores.id
    LEFT JOIN
        countries
        ON
            transactions.country_id = countries.id
""")

joined_df.write.mode('overwrite').saveAsTable('transact_countries')


This is an improvement. Referring back to the Spark UI **Stages** tab, note that there are no large shuffle reads anymore.  Only the small tables were shuffled, but the large table was not, so we avoided moving 1.4GB two times.

Broadcast joins can be significantly faster than shuffle joins if one of the tables is very large and the other is small. Unfortunately, broadcast joins only work if at least one of the tables are less than 100MB in size. In case of joining bigger tables, if we want to avoid the shuffle we may need to reconsider our schema to avoid having to do the join in the first place.

## Aggregations

Aggregations also use a shuffle, but they're often much less expensive. The following cell exeuctes a query that demonstrates this.

In [0]:
%sql
SELECT 
  country_id, 
  COUNT(*) AS count,
  AVG(amount) AS avg_amount
  FROM transactions
  GROUP BY country_id
  ORDER BY count DESC

That was fast! There are a lot of things going on here. One of the main points is that we're only shuffling the counts and sums necessary to compute the counts and averages that were requested. This only results in shuffling a few KB. Use the Spark UI once again to verify this.

So the shuffle is cheap compared to the shuffle joins, which need to shuffle all of the data. It also helps that our output is basically 0 in this case.


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>